In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/nba_features.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df['date'].min(), "to", df['date'].max())
print(df[['team', 'date', 'opponent', 'result']].head(10))

In [ ]:
def calculate_elo(df, k=20, initial=1500):
    """
    Calculate ELO rating for every team before each game.
    k  = how fast ratings change (20 is standard)
    initial = starting ELO for all teams (1500 is standard)
    """
    
    elo = {}
    for team in df['team'].unique():
        elo[team] = initial

    team_elo_before = []  
    opp_elo_before  = []

    for _, row in df.iterrows():
        team = row['team']
        opp  = row['opponent']

        
        t_elo = elo.get(team, initial)
        o_elo = elo.get(opp,  initial)

        
        team_elo_before.append(t_elo)
        opp_elo_before.append(o_elo)

        
        expected = 1 / (1 + 10 ** ((o_elo - t_elo) / 400))
        actual   = row['result']  

        
        elo[team] = t_elo + k * (actual - expected)
        elo[opp]  = o_elo + k * ((1 - actual) - (1 - expected))

    df['team_elo']    = team_elo_before
    df['opp_elo']     = opp_elo_before
    df['elo_diff']    = df['team_elo'] - df['opp_elo']

    return df

df = calculate_elo(df)
print("ELO added!")
print(df[['team', 'date', 'team_elo', 'opp_elo', 'elo_diff', 'result']].head(10))

In [ ]:
def add_h2h(df):
    """Win rate of team vs this specific opponent historically."""
    h2h_rates = []

    for i, row in df.iterrows():
        team = row['team']
        opp  = row['opponent']
        date = row['date']

        
        past = df[
            (df['team'] == team) &
            (df['opponent'] == opp) &
            (df['date'] < date)
        ].tail(10)  

        rate = past['result'].mean() if len(past) > 0 else 0.5
        h2h_rates.append(rate)

        
        if i % 5000 == 0:
            print(f"  Processing row {i}/{len(df)}...")

    df['h2h_win_rate'] = h2h_rates
    return df

print("Calculating H2H rates (this takes 3-5 minutes)...")
df = add_h2h(df)
print("Done!")
print(df[['team', 'opponent', 'h2h_win_rate']].head(10))

In [ ]:
df.to_csv('data/processed/nba_final.csv', index=False)
print(f"Saved! {len(df)} rows, {len(df.columns)} columns")
print("\nFinal feature list:")
print(df.columns.tolist())